In [ ]:
# Falling cube

import mujoco
import mujoco.viewer
import time

xml_string = """
<mujoco>
    <worldbody>
        <light name="top" pos="0 0 3"/>
        <geom name="floor" type="plane" size="2 2 0.1" rgba="0.8 0.8 0.8 1"/>
        <body name="falling_block" pos="0 0 1.5">
            <freejoint/>
            <geom type="box" size="0.2 0.2 0.2" rgba="1 0 0 1"/>
        </body>
    </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml_string)
data = mujoco.MjData(model)

# 1. Open the passive viewer (non-blocking)
with mujoco.viewer.launch_passive(model, data) as viewer:

    # 2. Keep the loop alive as long as the window is open
    while viewer.is_running():
        step_start = time.time()

        # Step the physics environment forward
        mujoco.mj_step(model, data)

        # Update the visual frame inside the GUI window
        viewer.sync()

        # 3. Handle real-time synchronization so the physics don't run at super-speed
        time_until_next_step = model.opt.timestep - (time.time() - step_start)
        if time_until_next_step > 0:
            time.sleep(time_until_next_step)

In [ ]:
# 2-joint arm with an end-effector marker

import time
import mujoco
import mujoco.viewer
import numpy as np

# 1. Define a minimal arm model with a designated site for the end effector
xml_model = """
<mujoco>
    <worldbody>
        <light pos="0 0 3"/>
        <body name="arm_base" pos="0 0 0">
            <joint name="joint1" type="hinge" axis="0 1 0"/>
            <geom type="cylinder" size="0.05 0.3" rgba="0.7 0.7 0.7 1"/>
            <body name="forearm" pos="0 0 0.6">
                <joint name="joint2" type="hinge" axis="0 1 0"/>
                <geom type="cylinder" size="0.04 0.25" rgba="0.5 0.5 0.5 1"/>
                <site name="end_effector" pos="0 0 0.5" size="0.01"/>
            </body>
        </body>
    </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(xml_model)
data = mujoco.MjData(model)

# Find the ID of the end-effector site
ee_site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "end_effector")

# 2. Launch the passive viewer loop
with mujoco.viewer.launch_passive(model, data) as viewer:
    while viewer.is_running():
        # Feed arbitrary moving joint values over time to test FK
        t = time.time()
        data.qpos[0] = np.sin(t)
        data.qpos[1] = np.cos(t)

        # Compute Forward Kinematics 
        mujoco.mj_forward(model, data)

        # Extract the global Cartesian coordinates of the end effector site
        ee_pos = data.site_xpos[ee_site_id]

        # 3. Inject the custom visual marker into the user scene
        viewer.user_scn.ngeom = 0  # Clear previous frame's markers

        mujoco.mjv_initGeom(
            viewer.user_scn.geoms[0],
            type=mujoco.mjtGeom.mjGEOM_SPHERE,
            size=[0.03, 0, 0],         # Radius of 3cm
            pos=ee_pos,
            mat=np.eye(3).flatten(),   # Global orientation matrix alignment
            rgba=[1, 0, 0, 1]          # Bright red, fully opaque
        )
        viewer.user_scn.ngeom = 1      # Let the viewer know 1 custom geom is ready

        # Sync modifications back to the UI thread
        viewer.sync()
        time.sleep(0.01)